## Partie A : Les données

In [79]:
import numpy as np

In [80]:
observations = [
    [("B", "T", "G"), ("B", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("D", "T", "P")],
    [("B", "T", "G"), ("B", "T", "G"), ("D", "T", "P")],
    [("B", "S", "G"), ("B", "S", "G"), ("D", "S", "P")],
    [("B", "S", "G"), ("B", "S", "G"), ("B", "S", "P")],
    [("D", "S", "G"), ("D", "S", "G"), ("D", "S", "P")]
]
print(observations)

[[('B', 'T', 'G'), ('B', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('D', 'T', 'P')], [('B', 'T', 'G'), ('B', 'T', 'G'), ('D', 'T', 'P')], [('B', 'S', 'G'), ('B', 'S', 'G'), ('D', 'S', 'P')], [('B', 'S', 'G'), ('B', 'S', 'G'), ('B', 'S', 'P')], [('D', 'S', 'G'), ('D', 'S', 'G'), ('D', 'S', 'P')]]


In [81]:
def generate_alea(n):
    """
    - C = Cheveux ∈ {B, D}  (Blond, Dark)
    - H = Hauteur ∈ {T, S}  (Tall, Short)
    - P = Pays ∈ {G, P}     (Greenland, Poland)
    """
    tableau = []
    for i in range(n):
        ligne = []
        for j in range(3):
            x1 = np.random.choice(['B', 'D'])
            x2 = np.random.choice(['T', 'S'])
            x3 = np.random.choice(['G', 'P'])
            ligne.append((x1,x2,x3))
        tableau.append(ligne)
    return tableau
tableau_alea = generate_alea(3)
print(tableau_alea)

[[('D', 'T', 'P'), ('D', 'T', 'G'), ('B', 'S', 'P')], [('B', 'T', 'G'), ('B', 'S', 'G'), ('D', 'T', 'G')], [('B', 'S', 'G'), ('B', 'T', 'G'), ('B', 'T', 'G')]]


## Partie B : Question

*"Si vous observez un nouvel individu grand ayant cheveux blonds, quel est son pays d’origine le plus probable?"*

-> Il s'agit de comparer P(Greenland|B,T) avec P(Poland|B,T)

In [82]:
def generate_DTG(n):
    table = []
    choix = ['D','T','G']
    for _ in range(n):
        triplet = np.random.choice(choix, size=3, replace=False)
        triplet_liste = list(triplet)
        table.append(triplet_liste)
    return table
print(generate_DTG(10))

[['G', 'T', 'D'], ['G', 'T', 'D'], ['T', 'G', 'D'], ['G', 'T', 'D'], ['D', 'G', 'T'], ['T', 'D', 'G'], ['D', 'G', 'T'], ['D', 'T', 'G'], ['T', 'D', 'G'], ['D', 'T', 'G']]


## Partie C : Maximum a posteriori

### 1. MAP dans le cadre Naïf de Bayes

In [83]:
def map_naive_bayes(data, query):
    cheveux, hauteur = query
    
    total_g = 0; compte_c_g = 0; compte_h_g = 0
    total_p = 0; compte_c_p = 0; compte_h_p = 0
    total = 0
    
    for row in data:
        for triplet in row:
            total += 1
            c_obs, h_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if c_obs == cheveux: compte_c_g += 1
                if h_obs == hauteur: compte_h_g += 1
            elif pays_obs == 'P':
                total_p += 1
                if c_obs == cheveux: compte_c_p += 1
                if h_obs == hauteur: compte_h_p += 1
                
    p_g = total_g / total if total > 0 else 0
    p_p = total_p / total if total > 0 else 0
    
    p_c_g = compte_c_g / total_g if total_g > 0 else 0
    p_h_g = compte_h_g / total_g if total_g > 0 else 0
    p_c_p = compte_c_p / total_p if total_p > 0 else 0
    p_h_p = compte_h_p / total_p if total_p > 0 else 0
    
    p_evidence_g = p_c_g * p_h_g
    p_evidence_p = p_c_p * p_h_p
    
    score_g = p_evidence_g * p_g
    score_p = p_evidence_p * p_p
    
    return {
        'P(B|G)': p_c_g,
        'P(T|G)': p_h_g,
        'P(B,T|G)': p_evidence_g,
        'P(G)': p_g,
        'Score(G)': score_g,
        'P(B|P)': p_c_p,
        'P(T|P)': p_h_p,
        'P(B,T|P)': p_evidence_p,
        'P(P)': p_p,
        'Score(P)': score_p,
        'Prediction': 'Greenland' if score_g > score_p else 'Poland'
    }

In [86]:
print("RÉSULTATS MAP NAÏF")
resultat = map_naive_bayes(tableau_alea, ('B', 'T'))
for cle, valeur in resultat.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

RÉSULTATS MAP NAÏF
P(B|G) : 0.7143
P(T|G) : 0.7143
P(B,T|G) : 0.5102
P(G) : 0.7778
Score(G) : 0.3968
P(B|P) : 0.5000
P(T|P) : 0.5000
P(B,T|P) : 0.2500
P(P) : 0.2222
Score(P) : 0.0556
Prediction : Greenland


### 2. MAP sans le cadre Naïf de Bayes

In [87]:
def map_no_naive_bayes(data, query):
    cheveux, hauteur = query
    
    total_g = 0; compte_b_t_g = 0
    total_p = 0; compte_b_t_p = 0
    total = 0
    
    for row in data:
        for triplet in row:
            total += 1
            c_obs, h_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if c_obs == cheveux and h_obs == hauteur: compte_b_t_g += 1
            elif pays_obs == 'P':
                total_p += 1
                if c_obs == cheveux and h_obs == hauteur: compte_b_t_p += 1

    p_g = total_g / total if total > 0 else 0
    p_p = total_p / total if total > 0 else 0
    
    p_evidence_g = compte_b_t_g / total_g if total_g > 0 else 0
    p_evidence_p = compte_b_t_p / total_p if total_p > 0 else 0
    
    score_g = p_evidence_g * p_g
    score_p = p_evidence_p * p_p
    
    return {
            'P(B,T|G)': p_evidence_g,
            'P(G)': p_g,
            'Score(G)': score_g,
            'P(B,T|P)': p_evidence_p,
            'P(P)': p_p,
            'Score(P)': score_p,
            'Count(B,T in G)': compte_b_t_g,
            'Count(B,T in P)': compte_b_t_p,
            'Prediction': 'Greenland' if score_g > score_p else 'Poland'
        }

In [88]:
print("RÉSULTATS MAP SANS NAÏF")
resultat_sans_naif = map_no_naive_bayes(tableau_alea, ('B', 'T'))
for cle, valeur in resultat_sans_naif.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

RÉSULTATS MAP SANS NAÏF
P(B,T|G) : 0.4286
P(G) : 0.7778
Score(G) : 0.3333
P(B,T|P) : 0.0000
P(P) : 0.2222
Score(P) : 0.0000
Count(B,T in G) : 3
Count(B,T in P) : 0
Prediction : Greenland


## Partie D : Maximum de vraisemblance

MLE Sans Naïf (Conjoint direct) : $$Vraisemblance = P(Cheveux, Hauteur \vert{} Pays)$$

MLE Naïf (Indépendance) : $$Vraisemblance = P(Cheveux \vert{} Pays) \times P(Hauteur \vert{} Pays)$$

#### MLE avec le cadre Naïf Bayes

In [100]:
def mle_naive_bayes(data, query):
    """
    MLE avec hypothèse Naïve Bayes
    Compare P(B|G)*P(T|G) avec P(B|P)*P(T|P)
    """
    cheveux, hauteur = query
    
    total_g = 0; compte_c_g = 0; compte_h_g = 0
    total_p = 0; compte_c_p = 0; compte_h_p = 0
    
    for row in data:
        for triplet in row:
            c_obs, h_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if c_obs == cheveux: compte_c_g += 1
                if h_obs == hauteur: compte_h_g += 1
                    
            elif pays_obs == 'P':
                total_p += 1
                if c_obs == cheveux: compte_c_p += 1
                if h_obs == hauteur: compte_h_p += 1
                
    # Vraisemblances individuelles
    p_c_g = compte_c_g / total_g if total_g > 0 else 0
    p_h_g = compte_h_g / total_g if total_g > 0 else 0
    p_c_p = compte_c_p / total_p if total_p > 0 else 0
    p_h_p = compte_h_p / total_p if total_p > 0 else 0
    
    # Hypothèse naïve : on multiplie les probabilités
    likelihood_g = p_c_g * p_h_g
    likelihood_p = p_c_p * p_h_p
    
    return {
        'P(B|G)': p_c_g, 'P(T|G)': p_h_g, 'P(B,T|G)': likelihood_g,
        'P(B|P)': p_c_p, 'P(T|P)': p_h_p, 'P(B,T|P)': likelihood_p,
        'Prediction': 'Greenland' if likelihood_g > likelihood_p else 'Poland'
    }

In [101]:
print("Naïf      :", mle_naive_bayes(observations, ('B', 'T')))

Naïf      : {'P(B|G)': 0.5, 'P(T|G)': 0.625, 'P(B,T|G)': 0.3125, 'P(B|P)': 0.5, 'P(T|P)': 0.625, 'P(B,T|P)': 0.3125, 'Prediction': 'Poland'}


#### MLE sans le cadre Naîf Bayes

In [93]:
def mle_no_naive_bayes(data, query):
    """
    MLE sans hypothèse Naïve Bayes
    Compare directement P(B,T|G) avec P(B,T|P)
    """
    cheveux, hauteur = query
    
    # Initialisation des compteurs globaux (dénominateurs)
    total_g = 0
    total_p = 0
    
    # Initialisation des compteurs de la requête exacte (numérateurs)
    compte_b_t_g = 0
    compte_b_t_p = 0
    
    # Parcours imbriqué unique : très optimisé et lisible
    for row in data:
        for triplet in row:
            # On extrait les valeurs pour simplifier la lecture
            c_obs, h_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                # Si l'individu est du Groenland, on vérifie s'il correspond au profil exact
                if c_obs == cheveux and h_obs == hauteur:
                    compte_b_t_g += 1
                    
            elif pays_obs == 'P':
                total_p += 1
                # Si l'individu est de Pologne, on vérifie s'il correspond au profil exact
                if c_obs == cheveux and h_obs == hauteur:
                    compte_b_t_p += 1
                    
    # Calcul des vraisemblances P(B,T|Pays)
    likelihood_g = compte_b_t_g / total_g if total_g > 0 else 0
    likelihood_p = compte_b_t_p / total_p if total_p > 0 else 0
    
    return {
        'P(B,T|G)': likelihood_g,
        'P(B,T|P)': likelihood_p,
        'Count(B,T in G)': compte_b_t_g,
        'Count(B,T in P)': compte_b_t_p,
        'Prediction': 'Greenland' if likelihood_g > likelihood_p else 'Poland'
    }


In [102]:
print("non Naïf      :", mle_no_naive_bayes(observations, ('B', 'T')))

non Naïf      : {'P(B,T|G)': 0.25, 'P(B,T|P)': 0.375, 'Count(B,T in G)': 4, 'Count(B,T in P)': 3, 'Prediction': 'Poland'}


## Partie E :